# 5 · Hyperparameter Tuning

A **controlled** study: one control run, then one probe per factor, all at the same budget, seed, data and batch size - so any difference is attributable to the single factor that moved.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # import the project's src/ package
import pandas as pd
from IPython.display import Image, display
from src import paths


In [ ]:
log = pd.read_csv(paths.TRAINING_OUTPUT_DIR / 'experiment_log.csv')
probes = log[log.experiment_id.str.startswith('e4')]
probes[['experiment_id','optimizer','lr0','epochs_run','precision','recall','map50','map50_95','notes']]

### Effect of each change, measured against the control

In [ ]:
ctrl = probes[probes.experiment_id == 'e4d_probe_baseline'].iloc[0]
for _, r in probes[probes.experiment_id != 'e4d_probe_baseline'].iterrows():
    d = r['map50_95'] - ctrl['map50_95']
    print(f"{r['experiment_id']:22s} mAP50-95 {r['map50_95']:.4f}  "
          f"({d:+.4f} vs control)  ->  {'better' if d > 0 else 'not better'}")

**Caveat we state out loud:** short probes rank configurations *under a short schedule*. The learning-rate schedule is a function of total epochs, so a setting that wins at 5 epochs is not guaranteed to win at 50. With more compute the honest design repeats the probes at full length.

## Final model (E5)

Continued fine-tuning of the strongest checkpoint using the winning configuration.

In [ ]:
log[log.experiment_id == 'e5_final'].T